# z010 - ¿Más árboles al entrenar con más meses?

**Idea:** Optuna eligió 538 árboles entrenando con 1 mes. El modelo final entrena con 4 meses:
con más datos puede convenir más árboles. Para medirlo hace falta validar 2 meses después
del último mes de train → train **202103 + 202104**, valid 202106.

Se compara ×1 (538), ×1,5 (807) y ×2 (1076) árboles. Además, como referencia, train solo 202104 con 538.
Se entrena una vez con 1076 árboles y se predice usando los primeros 538 / 807 / 1076.

In [1]:
# --- 1. Librerías y parámetros comunes ---
import numpy as np
import pandas as pd
import lightgbm as lgb

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   = 27_500
MES_VALID = 202106
SEMILLAS  = [501419, 623353, 786349, 834569, 946331]
N_PARTICIONES = 50        # Shuffle Split del test: particiones al azar del 50% de 202106

# Trial 34 de z005 (el mejor en promedio de 5 semillas)
params = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
    "num_leaves": 137, "min_sum_hessian_in_leaf": 9.755574,
}
ARBOLES = 538

In [2]:
# --- 2. Cargar datos y lags (igual que z003) ---
df = pd.read_parquet("../competencia_01.parquet")
df["clase_binaria"] = (df["clase_ternaria"] == "BAJA+2").astype(int)

no_usar = ["foto_mes", "clase_ternaria", "clase_binaria", "clase_binaria2"]
cols = [c for c in df.columns if c not in no_usar and c != "numero_de_cliente"]

df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
grupo = df.groupby("numero_de_cliente")[cols]
nuevas = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    nuevas += [lag, delta]
df = pd.concat([df] + nuevas, axis=1)
del nuevas, lag, delta, grupo

variables = [c for c in df.columns if c not in no_usar]
valid = df[df["foto_mes"] == MES_VALID]
# La ganancia SIEMPRE se calcula con BAJA+2 (aunque se entrene con otro target)
ganancia_cliente = np.where(valid["clase_ternaria"].values == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO)
print("Variables:", len(variables))

Variables: 761


In [3]:
# --- 3. Funciones de evaluación ---
def metricas(prob, mascara=None):
    # Ganancia acumulada ordenando de mayor a menor probabilidad (opcional: solo una parte de los clientes)
    g = ganancia_cliente if mascara is None else ganancia_cliente[mascara]
    p = prob if mascara is None else prob[mascara]
    acum = np.cumsum(g[np.argsort(p)[::-1]])
    n = int(np.argmax(acum)) + 1
    return acum[n - 1] / 1e6, n, acum[min(11_999, len(acum) - 1)] / 1e6

def resumen(probs_por_version):
    """probs_por_version = {version: [prob_semilla1, ..., prob_semilla5]}"""
    # a) Por semilla: ganancia máxima, mejor N y top 12.000
    filas = []
    for v, lista in probs_por_version.items():
        for s, pr in zip(SEMILLAS, lista):
            g, n, g12 = metricas(pr)
            filas.append({"version": v, "semilla": s, "ganancia_max_M": round(g, 1), "mejor_n": n, "top12000_M": round(g12, 1)})
    por_semilla = pd.DataFrame(filas)
    tabla = por_semilla.groupby("version", sort=False)[["ganancia_max_M", "top12000_M", "mejor_n"]].agg(["mean", "std"]).round(1)

    # b) Shuffle Split del test: ensamble de las 5 semillas evaluado en 50 particiones del 50% de 202106
    rng = np.random.default_rng(SEMILLAS[0])
    ens = {v: np.mean(lista, axis=0) for v, lista in probs_por_version.items()}
    part = []
    for i in range(N_PARTICIONES):
        mascara = rng.random(len(ganancia_cliente)) < 0.5
        part.append({v: metricas(pr, mascara)[0] for v, pr in ens.items()})
    part = pd.DataFrame(part)
    tabla[("shuffle_split", "mean")] = part.mean().round(1)
    tabla[("shuffle_split", "std")] = part.std().round(1)
    tabla[("shuffle_split", "veces_mejor")] = part.idxmax(axis=1).value_counts().reindex(tabla.index).fillna(0).astype(int)
    tabla[("ensamble_5", "ganancia_max_M")] = [round(metricas(ens[v])[0], 1) for v in tabla.index]
    return por_semilla, tabla

def anotar(tabla, prefijo, meses_train, extra=""):
    ruta_res = "resultados_experimentos.csv"
    res = pd.read_csv(ruta_res, sep=";", decimal=",", encoding="utf-8-sig")
    res = res[~res["descripcion"].str.startswith(prefijo)]   # sin duplicar si se vuelve a correr
    for v, r in tabla.iterrows():
        nueva = {
            "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
            "descripcion": (f"{prefijo} {v}: prom. {len(SEMILLAS)} semillas; shuffle split {r[('shuffle_split', 'mean')]} "
                            f"(mejor en {r[('shuffle_split', 'veces_mejor')]}/{N_PARTICIONES}){extra}"),
            "meses_train": str(meses_train), "mes_valid": MES_VALID, "semilla": str(SEMILLAS),
            "params": str(params), "mejor_n": int(r[("mejor_n", "mean")]),
            "ganancia_max_M": r[("ganancia_max_M", "mean")], "ganancia_top12000_M": r[("top12000_M", "mean")],
        }
        res = pd.concat([res, pd.DataFrame([nueva])], ignore_index=True)
    res["n_enviados"] = res["n_enviados"].astype("Int64")
    res.to_csv(ruta_res, sep=";", decimal=",", index=False, encoding="utf-8-sig")
    print("Anotado en", ruta_res)

In [4]:
# --- 4. Entrenar ---
probs = {"202104, 538 árboles (referencia)": []}
for k in [1, 1.5, 2]:
    probs[f"202103-202104, {int(ARBOLES * k)} árboles (x{k})"] = []

train1 = df[df["foto_mes"] == 202104]
train2 = df[df["foto_mes"].isin([202103, 202104])]
datos1 = lgb.Dataset(train1[variables], label=train1["clase_binaria"], free_raw_data=False)
datos2 = lgb.Dataset(train2[variables], label=train2["clase_binaria"], free_raw_data=False)

for s in SEMILLAS:
    probs["202104, 538 árboles (referencia)"].append(
        lgb.train({**params, "seed": s}, datos1, num_boost_round=ARBOLES).predict(valid[variables]))
    modelo = lgb.train({**params, "seed": s}, datos2, num_boost_round=ARBOLES * 2)
    for k in [1, 1.5, 2]:
        probs[f"202103-202104, {int(ARBOLES * k)} árboles (x{k})"].append(
            modelo.predict(valid[variables], num_iteration=int(ARBOLES * k)))
    print("Semilla", s, "lista")

Semilla 501419 lista


Semilla 623353 lista


Semilla 786349 lista


Semilla 834569 lista


Semilla 946331 lista


In [5]:
# --- 5. Comparación ---
por_semilla, tabla = resumen(probs)
print(por_semilla.pivot(index="semilla", columns="version", values="ganancia_max_M"))
tabla

version  202103-202104, 1076 árboles (x2)  202103-202104, 538 árboles (x1)  \
semilla                                                                      
501419                              392.7                            401.3   
623353                              394.8                            408.8   
786349                              396.9                            411.5   
834569                              397.3                            401.1   
946331                              401.6                            406.3   

version  202103-202104, 807 árboles (x1.5)  202104, 538 árboles (referencia)  
semilla                                                                       
501419                               394.9                             400.5  
623353                               407.3                             397.2  
786349                               400.2                             401.4  
834569                               397.9                

ganancia_max_M      top12000_M       \
                                            mean  std       mean  std   
version                                                                 
202104, 538 árboles (referencia)           399.6  2.3      392.9  3.7   
202103-202104, 538 árboles (x1)            405.8  4.6      397.1  2.8   
202103-202104, 807 árboles (x1.5)          400.5  4.7      394.5  3.4   
202103-202104, 1076 árboles (x2)           396.7  3.3      389.4  4.3   

                                   mejor_n         shuffle_split        \
                                      mean     std          mean   std   
version                                                                  
202104, 538 árboles (referencia)   13212.0  1072.1         199.1  11.6   
202103-202104, 538 árboles (x1)    13396.2   367.2         202.3  12.5   
202103-202104, 807 árboles (x1.5)  13690.6   613.2         200.3  12.9   
202103-202104, 1076 árboles (x2)   13087.0   858.9         197.1  12.9   

                                                  ensamble_5  
                                  veces_mejor ganancia_max_M  
version                                                       
202104, 538 árboles (referencia)            8          402.4  
202103-202104, 538 árboles (x1)            33          409.4  
202103-202104, 807 árboles (x1.5)           9          406.2  
202103-202104, 1076 árboles (x2)            0          398.9

In [6]:
# --- 6. Anotar ---
anotar(tabla, "z010 arboles", [202103, 202104])

Anotado en resultados_experimentos.csv


## Conclusiones (08/10)
- Train 202103-202104 con los mismos 538 árboles: **405,8 M** (+6 M vs 1 mes); ×1,5: 400,5; ×2: 396,7.
- **No conviene subir los árboles** al sumar meses: con 2 meses, 538 ya es lo mejor.
- Decisión: el ensamble final se queda con el best_iter de Optuna.